### Source Tables:
- _exponent._bronze_allscripts_tw_works.dbo_item_result (measurement header - 413M records)
- _exponent._bronze_allscripts_tw_works.dbo_result (measurement values/results - 427M records)
- _exponent._bronze_allscripts_tw_works.dbo_item_finding (findings header - 283M records)
- _exponent._bronze_allscripts_tw_works.dbo_finding (finding values - 298M records)

### Notes:
- PERSON and VISIT_OCCURRENCE must run before MEASUREMENT
- dbo_item_result.ID is the measurement identifier
- dbo_item_result.CurrentID links to dbo_result.ID for actual values
- dbo_item_result.PatientID links to dbo_person.ID
- Filters for NumericResult IS NOT NULL (quantitative data only)
- QODE = measurement type code (links to dbo_qo_de.ID)
- LOINC codes available in RIDLOINCCodeList column
- Starting with dbo_item_result/dbo_result (can add findings later)

# Transformation

In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_allscripts.measurement;

In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_tw.measurement;

In [0]:
%sql
-- DELETE FROM _exponent.omop_silver.measurement
-- WHERE source_system = 'allscripts_tw';

In [0]:
%sql
-- DELETE FROM _exponent.omop_mapping.source_to_measurement
-- WHERE source_system = 'allscripts_tw';

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_measurement_findings AS
WITH unit_code_mapping AS (
  SELECT
    dbo_unit_code_de.id,
    dbo_unit_code_de.entrycode AS unit_source_value,
    CASE
      WHEN TRIM(dbo_unit_code_de.normalizedunits) = 'percent' THEN '%'
      WHEN TRIM(dbo_unit_code_de.normalizedunits) = 'mm Hg' THEN 'mm[Hg]'
      WHEN TRIM(dbo_unit_code_de.normalizedunits) = 'oC' THEN 'Cel'
      WHEN TRIM(dbo_unit_code_de.normalizedunits) = 'ml' THEN 'mL'
      WHEN TRIM(dbo_unit_code_de.normalizedunits) = 'seconds' THEN 's'
      WHEN TRIM(dbo_unit_code_de.normalizedunits) = 'ratio' THEN '{ratio}'
      ELSE TRIM(dbo_unit_code_de.normalizedunits)
    END AS mapped_ucum_code
  FROM _exponent._bronze_allscripts_tw_works_vw.dbo_unit_code_de
)

SELECT
  source_to_person.person_id,
  concept.concept_id AS measurement_concept_id,
  CAST(dbo_item_finding.performeddttm AS DATE) AS measurement_date,
  dbo_item_finding.performeddttm AS measurement_datetime,
  NULL AS measurement_time,
  32817 AS measurement_type_concept_id,
  NULL AS operator_concept_id,
  COALESCE(dbo_finding.normalizedvalue, dbo_finding.numericfinding) AS value_as_number,
  NULL AS value_as_concept_id,
  COALESCE(unit_concept.concept_id, 0) AS unit_concept_id,
  NULL AS range_low,
  NULL AS range_high,
  NULL AS provider_id,
  source_to_visit_occurrence.visit_occurrence_id AS visit_occurrence_id,
  NULL AS visit_detail_id,
  CONCAT_WS(
    CHR(31),
    'allscripts_tw',
    'dbo_item_finding',
    'id',
    CAST(dbo_item_finding.id AS BIGINT)
  ) AS measurement_source_value,
  0 AS measurement_source_concept_id,
  unit_code_mapping.unit_source_value,
  0 AS unit_source_concept_id,
  CAST(COALESCE(dbo_finding.normalizedvalue, dbo_finding.numericfinding) AS STRING) AS value_source_value,
  NULL AS measurement_event_id,
  NULL AS meas_event_field_concept_id,
  'allscripts_tw' AS source_system
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_item_finding
JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_finding
  ON dbo_finding.id = dbo_item_finding.currentid
JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_act_hdr_other
  ON dbo_act_hdr_other.id = dbo_item_finding.activityheaderid
JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_visit
  ON dbo_visit.id = dbo_act_hdr_other.visitid
JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_person
  ON dbo_person.id = dbo_item_finding.patientid
JOIN _exponent.omop_mapping.source_to_person
  ON source_to_person.person_source_value =
     CONCAT_WS(CHAR(31), 'allscripts_tw', 'dbo_person', 'id', CAST(dbo_person.id AS BIGINT))
JOIN _exponent.omop_mapping.source_to_visit_occurrence
  ON source_to_visit_occurrence.visit_occurrence_source_value =
     CONCAT_WS(CHAR(31), 'allscripts_tw', 'dbo_visit', 'id', CAST(dbo_visit.id AS BIGINT))
 AND source_to_person.active_flag = TRUE
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_qo_de
  ON dbo_item_finding.qode = dbo_qo_de.id
LEFT JOIN unit_code_mapping
  ON dbo_qo_de.unitsde = unit_code_mapping.id
LEFT JOIN _exponent.omop.concept
  ON concept.vocabulary_id = 'LOINC'
 AND concept.concept_code = TRIM(dbo_qo_de.loinclabcode)
LEFT JOIN _exponent.omop.concept unit_concept
  ON unit_concept.vocabulary_id = 'UCUM'
 AND unit_concept.concept_code = unit_code_mapping.mapped_ucum_code
WHERE dbo_finding.answerdatatypede = 3
  AND COALESCE(dbo_finding.normalizedvalue, dbo_finding.numericfinding) IS NOT NULL
  AND COALESCE(dbo_finding.normalizedvalue, dbo_finding.numericfinding) <> 0
  AND LOWER(dbo_qo_de.entryname) NOT LIKE '%percentile%'
  AND dbo_qo_de.loinclabcode IS NOT NULL
  AND TRIM(dbo_qo_de.loinclabcode) <> ''
  AND TRIM(dbo_qo_de.loinclabcode) RLIKE '^[0-9]+-[0-9]+$'
  AND concept.domain_id = 'Measurement';


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_measurement_results AS
WITH unit_code_mapping AS (
  SELECT
    dbo_unit_code_de.id,
    dbo_unit_code_de.entrycode AS unit_source_value,
    CASE
      WHEN TRIM(dbo_unit_code_de.normalizedunits) = 'percent' THEN '%'
      WHEN TRIM(dbo_unit_code_de.normalizedunits) = 'mm Hg' THEN 'mm[Hg]'
      WHEN TRIM(dbo_unit_code_de.normalizedunits) = 'oC' THEN 'Cel'
      WHEN TRIM(dbo_unit_code_de.normalizedunits) = 'ml' THEN 'mL'
      WHEN TRIM(dbo_unit_code_de.normalizedunits) = 'seconds' THEN 's'
      WHEN TRIM(dbo_unit_code_de.normalizedunits) = 'ratio' THEN '{ratio}'
      ELSE TRIM(dbo_unit_code_de.normalizedunits)
    END AS mapped_ucum_code
  FROM _exponent._bronze_allscripts_tw_works_vw.dbo_unit_code_de
)

SELECT
  source_to_person.person_id,
  concept.concept_id AS measurement_concept_id,
  CAST(COALESCE(dbo_result.clinicaldttm, dbo_result.performeddttm, dbo_item_result.performeddttm) AS DATE) AS measurement_date,
  COALESCE(dbo_result.clinicaldttm, dbo_result.performeddttm, dbo_item_result.performeddttm) AS measurement_datetime,
  NULL AS measurement_time,
  32817 AS measurement_type_concept_id,
  NULL AS operator_concept_id,
  COALESCE(dbo_result.normalizedvalue, dbo_result.numericresult) AS value_as_number,
  NULL AS value_as_concept_id,
  COALESCE(unit_concept.concept_id, 0) AS unit_concept_id,
  NULL AS range_low,
  NULL AS range_high,
  NULL AS provider_id,
  source_to_visit_occurrence.visit_occurrence_id AS visit_occurrence_id,
  NULL AS visit_detail_id,
  CONCAT_WS(
    CHR(31),
    'allscripts_tw',
    'dbo_item_result',
    'id',
    CAST(dbo_item_result.id AS BIGINT)
  ) AS measurement_source_value,
  0 AS measurement_source_concept_id,
  COALESCE(unit_code_mapping.unit_source_value, dbo_result.unitsdet) AS unit_source_value,
  0 AS unit_source_concept_id,
  CAST(COALESCE(dbo_result.normalizedvalue, dbo_result.numericresult) AS STRING) AS value_source_value,
  NULL AS measurement_event_id,
  NULL AS meas_event_field_concept_id,
  'allscripts_tw' AS source_system
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_item_result
JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_result
  ON dbo_result.id = dbo_item_result.currentid
JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_person
  ON dbo_person.id = dbo_item_result.patientid
JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_act_hdr_result
  ON dbo_act_hdr_result.id = dbo_item_result.activityheaderid
JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_visit
  ON dbo_visit.id = dbo_act_hdr_result.visitid
JOIN _exponent.omop_mapping.source_to_person
  ON source_to_person.person_source_value =
     CONCAT_WS(CHAR(31), 'allscripts_tw', 'dbo_person', 'id', CAST(dbo_person.id AS BIGINT))
 AND source_to_person.active_flag = TRUE
JOIN _exponent.omop_mapping.source_to_visit_occurrence
  ON source_to_visit_occurrence.visit_occurrence_source_value =
     CONCAT_WS(CHAR(31), 'allscripts_tw', 'dbo_visit', 'id', CAST(dbo_visit.id AS BIGINT))
 AND source_to_person.active_flag = TRUE
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_qo_de
  ON dbo_item_result.qode = dbo_qo_de.id
LEFT JOIN unit_code_mapping
  ON dbo_qo_de.unitsde = unit_code_mapping.id
LEFT JOIN _exponent.omop.concept
  ON concept.vocabulary_id = 'LOINC'
 AND concept.concept_code = TRIM(dbo_qo_de.loinclabcode)
LEFT JOIN _exponent.omop.concept unit_concept
  ON unit_concept.vocabulary_id = 'UCUM'
 AND unit_concept.concept_code = unit_code_mapping.mapped_ucum_code
WHERE COALESCE(dbo_result.normalizedvalue, dbo_result.numericresult) IS NOT NULL
  AND COALESCE(dbo_result.normalizedvalue, dbo_result.numericresult) <> 0
  AND dbo_qo_de.loinclabcode IS NOT NULL
  AND TRIM(dbo_qo_de.loinclabcode) <> ''
  AND TRIM(dbo_qo_de.loinclabcode) RLIKE '^[0-9]+-[0-9]+$'
  AND concept.domain_id = 'Measurement';


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_measurement AS
SELECT * FROM silver_measurement_findings
UNION ALL
SELECT * FROM silver_measurement_results

In [0]:
%sql
MERGE INTO _exponent.omop_silver.measurement AS target
USING silver_measurement AS source
ON target.measurement_source_value = source.measurement_source_value

WHEN MATCHED AND NOT (
  target.person_id <=> source.person_id
  AND target.measurement_concept_id <=> source.measurement_concept_id
  AND target.measurement_date <=> source.measurement_date
  AND target.measurement_datetime <=> source.measurement_datetime
  AND target.measurement_time <=> source.measurement_time
  AND target.measurement_type_concept_id <=> source.measurement_type_concept_id
  AND target.operator_concept_id <=> source.operator_concept_id
  AND target.value_as_number <=> source.value_as_number
  AND target.value_as_concept_id <=> source.value_as_concept_id
  AND target.unit_concept_id <=> source.unit_concept_id
  AND target.range_low <=> source.range_low
  AND target.range_high <=> source.range_high
  AND target.provider_id <=> source.provider_id
  AND target.visit_occurrence_id <=> source.visit_occurrence_id
  AND target.visit_detail_id <=> source.visit_detail_id
  AND target.measurement_source_value <=> source.measurement_source_value
  AND target.measurement_source_concept_id <=> source.measurement_source_concept_id
  AND target.unit_source_value <=> source.unit_source_value
  AND target.unit_source_concept_id <=> source.unit_source_concept_id
  AND target.value_source_value <=> source.value_source_value
  AND target.measurement_event_id <=> source.measurement_event_id
  AND target.meas_event_field_concept_id <=> source.meas_event_field_concept_id
  AND target.source_system <=> source.source_system
)
THEN UPDATE SET
  target.person_id = source.person_id,
  target.measurement_concept_id = source.measurement_concept_id,
  target.measurement_date = source.measurement_date,
  target.measurement_datetime = source.measurement_datetime,
  target.measurement_time = source.measurement_time,
  target.measurement_type_concept_id = source.measurement_type_concept_id,
  target.operator_concept_id = source.operator_concept_id,
  target.value_as_number = source.value_as_number,
  target.value_as_concept_id = source.value_as_concept_id,
  target.unit_concept_id = source.unit_concept_id,
  target.range_low = source.range_low,
  target.range_high = source.range_high,
  target.provider_id = source.provider_id,
  target.visit_occurrence_id = source.visit_occurrence_id,
  target.visit_detail_id = source.visit_detail_id,
  target.measurement_source_value = source.measurement_source_value,
  target.measurement_source_concept_id = source.measurement_source_concept_id,
  target.unit_source_value = source.unit_source_value,
  target.unit_source_concept_id = source.unit_source_concept_id,
  target.value_source_value = source.value_source_value,
  target.measurement_event_id = source.measurement_event_id,
  target.meas_event_field_concept_id = source.meas_event_field_concept_id,
  target.source_system = source.source_system

WHEN NOT MATCHED THEN INSERT (
  person_id,
  measurement_concept_id,
  measurement_date,
  measurement_datetime,
  measurement_time,
  measurement_type_concept_id,
  operator_concept_id,
  value_as_number,
  value_as_concept_id,
  unit_concept_id,
  range_low,
  range_high,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  measurement_source_value,
  measurement_source_concept_id,
  unit_source_value,
  unit_source_concept_id,
  value_source_value,
  measurement_event_id,
  meas_event_field_concept_id,
  source_system
)
VALUES (
  source.person_id,
  source.measurement_concept_id,
  source.measurement_date,
  source.measurement_datetime,
  source.measurement_time,
  source.measurement_type_concept_id,
  source.operator_concept_id,
  source.value_as_number,
  source.value_as_concept_id,
  source.unit_concept_id,
  source.range_low,
  source.range_high,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.measurement_source_value,
  source.measurement_source_concept_id,
  source.unit_source_value,
  source.unit_source_concept_id,
  source.value_source_value,
  source.measurement_event_id,
  source.meas_event_field_concept_id,
  source.source_system
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_measurement (
    source_system,
    measurement_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    source.source_system,
    source.measurement_source_value,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    current_timestamp() AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT
        source_system,
        measurement_source_value
    FROM _exponent.omop_silver.measurement
) source
LEFT ANTI JOIN _exponent.omop_mapping.source_to_measurement target
  ON source.measurement_source_value = target.measurement_source_value;

In [0]:
%sql
MERGE INTO _exponent.omop_tw.measurement AS target
USING (
    SELECT
        mapping.measurement_id,
        source.person_id,
        source.measurement_concept_id,
        source.measurement_date,
        source.measurement_datetime,
        source.measurement_time,
        source.measurement_type_concept_id,
        source.operator_concept_id,
        source.value_as_number,
        source.value_as_concept_id,
        source.unit_concept_id,
        source.range_low,
        source.range_high,
        source.provider_id,
        source.visit_occurrence_id,
        source.visit_detail_id,
        source.measurement_source_value,
        source.measurement_source_concept_id,
        source.unit_source_value,
        source.unit_source_concept_id,
        source.value_source_value,
        source.measurement_event_id,
        source.meas_event_field_concept_id
    FROM _exponent.omop_silver.measurement AS source
    JOIN _exponent.omop_mapping.source_to_measurement AS mapping
      ON mapping.measurement_source_value = source.measurement_source_value
    WHERE source.source_system = 'allscripts_tw'
) AS source
ON target.measurement_id = source.measurement_id

WHEN MATCHED AND NOT (
    target.person_id <=> source.person_id
    AND target.measurement_concept_id <=> source.measurement_concept_id
    AND target.measurement_date <=> source.measurement_date
    AND target.measurement_datetime <=> source.measurement_datetime
    AND target.measurement_time <=> source.measurement_time
    AND target.measurement_type_concept_id <=> source.measurement_type_concept_id
    AND target.operator_concept_id <=> source.operator_concept_id
    AND target.value_as_number <=> source.value_as_number
    AND target.value_as_concept_id <=> source.value_as_concept_id
    AND target.unit_concept_id <=> source.unit_concept_id
    AND target.range_low <=> source.range_low
    AND target.range_high <=> source.range_high
    AND target.provider_id <=> source.provider_id
    AND target.visit_occurrence_id <=> source.visit_occurrence_id
    AND target.visit_detail_id <=> source.visit_detail_id
    AND target.measurement_source_value <=> source.measurement_source_value
    AND target.measurement_source_concept_id <=> source.measurement_source_concept_id
    AND target.unit_source_value <=> source.unit_source_value
    AND target.unit_source_concept_id <=> source.unit_source_concept_id
    AND target.value_source_value <=> source.value_source_value
    AND target.measurement_event_id <=> source.measurement_event_id
    AND target.meas_event_field_concept_id <=> source.meas_event_field_concept_id
)
THEN UPDATE SET
    target.person_id = source.person_id,
    target.measurement_concept_id = source.measurement_concept_id,
    target.measurement_date = source.measurement_date,
    target.measurement_datetime = source.measurement_datetime,
    target.measurement_time = source.measurement_time,
    target.measurement_type_concept_id = source.measurement_type_concept_id,
    target.operator_concept_id = source.operator_concept_id,
    target.value_as_number = source.value_as_number,
    target.value_as_concept_id = source.value_as_concept_id,
    target.unit_concept_id = source.unit_concept_id,
    target.range_low = source.range_low,
    target.range_high = source.range_high,
    target.provider_id = source.provider_id,
    target.visit_occurrence_id = source.visit_occurrence_id,
    target.visit_detail_id = source.visit_detail_id,
    target.measurement_source_value = source.measurement_source_value,
    target.measurement_source_concept_id = source.measurement_source_concept_id,
    target.unit_source_value = source.unit_source_value,
    target.unit_source_concept_id = source.unit_source_concept_id,
    target.value_source_value = source.value_source_value,
    target.measurement_event_id = source.measurement_event_id,
    target.meas_event_field_concept_id = source.meas_event_field_concept_id

WHEN NOT MATCHED THEN INSERT (
    measurement_id,
    person_id,
    measurement_concept_id,
    measurement_date,
    measurement_datetime,
    measurement_time,
    measurement_type_concept_id,
    operator_concept_id,
    value_as_number,
    value_as_concept_id,
    unit_concept_id,
    range_low,
    range_high,
    provider_id,
    visit_occurrence_id,
    visit_detail_id,
    measurement_source_value,
    measurement_source_concept_id,
    unit_source_value,
    unit_source_concept_id,
    value_source_value,
    measurement_event_id,
    meas_event_field_concept_id
)
VALUES (
    source.measurement_id,
    source.person_id,
    source.measurement_concept_id,
    source.measurement_date,
    source.measurement_datetime,
    source.measurement_time,
    source.measurement_type_concept_id,
    source.operator_concept_id,
    source.value_as_number,
    source.value_as_concept_id,
    source.unit_concept_id,
    source.range_low,
    source.range_high,
    source.provider_id,
    source.visit_occurrence_id,
    source.visit_detail_id,
    source.measurement_source_value,
    source.measurement_source_concept_id,
    source.unit_source_value,
    source.unit_source_concept_id,
    source.value_source_value,
    source.measurement_event_id,
    source.meas_event_field_concept_id
);

In [0]:
%sql
MERGE INTO _exponent.omop_allscripts.measurement AS target
USING (
    SELECT
        mapping.measurement_id,
        source.person_id,
        source.measurement_concept_id,
        source.measurement_date,
        source.measurement_datetime,
        source.measurement_time,
        source.measurement_type_concept_id,
        source.operator_concept_id,
        source.value_as_number,
        source.value_as_concept_id,
        source.unit_concept_id,
        source.range_low,
        source.range_high,
        source.provider_id,
        source.visit_occurrence_id,
        source.visit_detail_id,
        source.measurement_source_value,
        source.measurement_source_concept_id,
        source.unit_source_value,
        source.unit_source_concept_id,
        source.value_source_value,
        source.measurement_event_id,
        source.meas_event_field_concept_id
    FROM _exponent.omop_silver.measurement AS source
    JOIN _exponent.omop_mapping.source_to_measurement AS mapping
      ON mapping.measurement_source_value = source.measurement_source_value
    WHERE source.source_system = 'allscripts_tw'
) AS source
ON target.measurement_id = source.measurement_id

WHEN MATCHED AND NOT (
    target.person_id <=> source.person_id
    AND target.measurement_concept_id <=> source.measurement_concept_id
    AND target.measurement_date <=> source.measurement_date
    AND target.measurement_datetime <=> source.measurement_datetime
    AND target.measurement_time <=> source.measurement_time
    AND target.measurement_type_concept_id <=> source.measurement_type_concept_id
    AND target.operator_concept_id <=> source.operator_concept_id
    AND target.value_as_number <=> source.value_as_number
    AND target.value_as_concept_id <=> source.value_as_concept_id
    AND target.unit_concept_id <=> source.unit_concept_id
    AND target.range_low <=> source.range_low
    AND target.range_high <=> source.range_high
    AND target.provider_id <=> source.provider_id
    AND target.visit_occurrence_id <=> source.visit_occurrence_id
    AND target.visit_detail_id <=> source.visit_detail_id
    AND target.measurement_source_value <=> source.measurement_source_value
    AND target.measurement_source_concept_id <=> source.measurement_source_concept_id
    AND target.unit_source_value <=> source.unit_source_value
    AND target.unit_source_concept_id <=> source.unit_source_concept_id
    AND target.value_source_value <=> source.value_source_value
    AND target.measurement_event_id <=> source.measurement_event_id
    AND target.meas_event_field_concept_id <=> source.meas_event_field_concept_id
)
THEN UPDATE SET
    target.person_id = source.person_id,
    target.measurement_concept_id = source.measurement_concept_id,
    target.measurement_date = source.measurement_date,
    target.measurement_datetime = source.measurement_datetime,
    target.measurement_time = source.measurement_time,
    target.measurement_type_concept_id = source.measurement_type_concept_id,
    target.operator_concept_id = source.operator_concept_id,
    target.value_as_number = source.value_as_number,
    target.value_as_concept_id = source.value_as_concept_id,
    target.unit_concept_id = source.unit_concept_id,
    target.range_low = source.range_low,
    target.range_high = source.range_high,
    target.provider_id = source.provider_id,
    target.visit_occurrence_id = source.visit_occurrence_id,
    target.visit_detail_id = source.visit_detail_id,
    target.measurement_source_value = source.measurement_source_value,
    target.measurement_source_concept_id = source.measurement_source_concept_id,
    target.unit_source_value = source.unit_source_value,
    target.unit_source_concept_id = source.unit_source_concept_id,
    target.value_source_value = source.value_source_value,
    target.measurement_event_id = source.measurement_event_id,
    target.meas_event_field_concept_id = source.meas_event_field_concept_id

WHEN NOT MATCHED THEN INSERT (
    measurement_id,
    person_id,
    measurement_concept_id,
    measurement_date,
    measurement_datetime,
    measurement_time,
    measurement_type_concept_id,
    operator_concept_id,
    value_as_number,
    value_as_concept_id,
    unit_concept_id,
    range_low,
    range_high,
    provider_id,
    visit_occurrence_id,
    visit_detail_id,
    measurement_source_value,
    measurement_source_concept_id,
    unit_source_value,
    unit_source_concept_id,
    value_source_value,
    measurement_event_id,
    meas_event_field_concept_id
)
VALUES (
    source.measurement_id,
    source.person_id,
    source.measurement_concept_id,
    source.measurement_date,
    source.measurement_datetime,
    source.measurement_time,
    source.measurement_type_concept_id,
    source.operator_concept_id,
    source.value_as_number,
    source.value_as_concept_id,
    source.unit_concept_id,
    source.range_low,
    source.range_high,
    source.provider_id,
    source.visit_occurrence_id,
    source.visit_detail_id,
    source.measurement_source_value,
    source.measurement_source_concept_id,
    source.unit_source_value,
    source.unit_source_concept_id,
    source.value_source_value,
    source.measurement_event_id,
    source.meas_event_field_concept_id
);